##  TechMind — Exploración y Preparación del Dataset **Microsoft Learn Catalog API**

#### Equipo tejONEs

#### 02_exploracion_dataset_mslearn.ipynb

💡**Dataset**: [Microsoft Learn Catalog - Extracción a traves de la API Oficial](https://learn.microsoft.com/en-us/training/support/catalog-api)

- El proceso general que sigue este pipeline es:
    - [x]  Extracción reproducible desde la API, carga y normalización al esquema del proyecto
    - [x]  Construcción de `texto` a partir de la descripción y el temario, y eliminación de registros finales en inglés
    - [x]  Limpieza de texto (HTML, URLs, duplicados y filtro de mínimo 100 palabras)
    - [x]  Análisis exploratorio de categorías, subcategorías y tecnologías mencionadas
    - [x]  Mapeo a las siete categorías mediante título, texto, tecnologías y subcategoría
    - [x]  Balanceo de datos (máximo 50 registros por categoría)
    - [x]  Validación del esquema, auditoría de calidad y exportación en `procesados/`

## Importaciones

In [ ]:
import os
import re
from pathlib import Path
import subprocess
import sys

import nltk
import pandas as pd
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
from tqdm.notebook import tqdm

# Descargar recursos de NLP
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))

## Configuración

In [2]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / 'data_science').exists() and (candidate / 'README.md').exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

CARPETA_DATA = str((project_root / 'data_science' / 'data').resolve())
CARPETA_CRUDOS = str((project_root / 'data_science' / 'data' / 'crudos').resolve())
CARPETA_PROCESADOS = str((project_root / 'data_science' / 'data' / 'procesados').resolve())


print(f'✅ Ruta de procesados existe: {Path(CARPETA_PROCESADOS).exists()}')
print(f'✅ Ruta de crudos existe: {Path(CARPETA_CRUDOS).exists()}')
print(f'✅ Ruta de datos existe: {Path(CARPETA_DATA).exists()}')
print(f'📂 Datos procesados: {CARPETA_PROCESADOS}')
print(f'📂 Datos crudos: {CARPETA_CRUDOS}')
print(f'📁 Proyecto local: {CARPETA_DATA}')

✅ Ruta de procesados existe: True
✅ Ruta de crudos existe: True
✅ Ruta de datos existe: True
📂 Datos procesados: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados
📂 Datos crudos: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\crudos
📁 Proyecto local: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data


# 1.Microsoft Learn API dataset

##  1.1 Carga de datos y normalización

### Extracción reproducible desde Microsoft Learn

La extracción desde Microsoft Learn Catalog API está implementada en [`mslearn_api.py`](../scripts/mslearn_api.py). Se mantiene separada para facilitar su reutilización, mantenimiento y ejecución independiente. La siguiente celda permite actualizar opcionalmente el archivo crudo.

La última ejecución de la extracción se realizó el **21 de julio de 2026** y dejó **4,606 filas** en `courses_mslearn.csv`.

In [ ]:
RUN_EXTRACTION = False  # Cambiar manualmente a True para actualizar los datos

extractor_path = (
    project_root / 'data_science' / 'scripts' / 'mslearn_api.py'
)
raw_path = Path(CARPETA_CRUDOS) / 'courses_mslearn.csv'

if RUN_EXTRACTION:
    subprocess.run(
        [
            sys.executable,
            str(extractor_path),
            '--limit', '300',
            '--out', str(raw_path),
        ],
        check=True,
    )
    print(f'✅ Extracción completada: {raw_path}')
else:
    print('ℹ️ Se utilizará el dataset crudo existente.')

In [3]:
file_path = f'{CARPETA_CRUDOS}/courses_mslearn.csv'

# Carga segura con alternativa de codificación (encoding fallback) en caso de error
try:
    df_mslearn = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df_mslearn = pd.read_csv(file_path, encoding='latin1')

print(f"✅ Datos cargados correctamente. Filas totales: {len(df_mslearn)}")
print("\n--- Muestra aleatoria de los datos cargados ---")
display(df_mslearn.sample(3))

✅ Datos cargados correctamente. Filas totales: 4606

--- Muestra aleatoria de los datos cargados ---


,id,titulo,descripcion,categoria,subcategoria,temario,tecnologias_mencionadas,tipo_contenido,nivel_dificultad,idioma,fuente,autor,url,fecha_publicacion,licencia,fecha_recoleccion
3755,3756,Implementación de la seguridad de administraci...,Aprenda cómo proteger recursos mediante direct...,Seguridad,NaN,Control y organización de los recursos de Azur...,"azure, azure portal, azure resource manager, b...",ruta de aprendizaje,Principiante,es,microsoft learn,Microsoft,https://learn.microsoft.com/es-es/training/pat...,2025-09-26,Información protegida (derechos de autor),2026-07-20
901,902,Desarrollo de un agente de voz en tiempo real ...,Aprenda a desarrollar un agente de voz en dire...,Inteligencia artificial,NaN,Introducción | Exploración de Azure Voice Live...,"api, azure, desarrollo, foundry-tools, microso...",módulo,Intermedio,es,microsoft learn,Microsoft,https://learn.microsoft.com/es-es/training/mod...,2026-06-11,Información protegida (derechos de autor),2026-07-20
1971,1972,Integración de agentes con sistemas externos a...,Conecte el agente de Copilot Studio a los serv...,Inteligencia artificial,NaN,Introducción | Descripción de MCP y su rol en ...,"api, copilot, microsoft, microsoft power platf...",módulo,Avanzado,es,microsoft learn,Microsoft,https://learn.microsoft.com/es-es/training/mod...,2026-05-11,Información protegida (derechos de autor),2026-07-20


In [4]:
df_mslearn.columns

Index(['id', 'titulo', 'descripcion', 'categoria', 'subcategoria', 'temario',
       'tecnologias_mencionadas', 'tipo_contenido', 'nivel_dificultad',
       'idioma', 'fuente', 'autor', 'url', 'fecha_publicacion', 'licencia',
       'fecha_recoleccion'],
      dtype='str')

Para ajustarse a la estructura del dataset con las columnas:
`titulo, texto, categoría, autor, tipo`

In [5]:
# Crear la columna 'texto' con descripcion concatenada a la columna 'temario'(limpiando los separadores '|')
df_mslearn['texto'] = df_mslearn['descripcion'] + ' ' + df_mslearn['temario'].str.replace('|', ' ', regex=False)


# Renombrar columnas para ajustarse al esquema
df_mslearn = df_mslearn.rename(columns={'tipo_contenido': 'tipo'})

# Seleccionar las columnas requeridas y mantener las auxiliares solicitadas
columnas_finales = ['titulo', 'texto','categoria', 'subcategoria', 'autor', 'tipo', 'tecnologias_mencionadas']
df_mslearn = df_mslearn[columnas_finales]

print("✅ Estructura ajustada correctamente.")
print(f"Columnas actuales: {df_mslearn.columns.tolist()}")
display(df_mslearn.head(3))

✅ Estructura ajustada correctamente.
Columnas actuales: ['titulo', 'texto', 'categoria', 'subcategoria', 'autor', 'tipo', 'tecnologias_mencionadas']


,titulo,texto,categoria,subcategoria,autor,tipo,tecnologias_mencionadas
0,Experimento con Azure Machine Learning,Obtenga información sobre cómo encontrar el me...,Inteligencia artificial,Aprendizaje automático,Microsoft,módulo,"azure, datos, ia, learning, machine, machine l..."
1,Implementación de la protección de datos del d...,En este módulo se describe cómo puede usar Int...,NaN,NaN,Microsoft,módulo,"configuration-manager, datos, intune, windows"
2,Adición de lógica de decisión al código median...,Aprenda a bifurcar la ruta de acceso de ejecuc...,Desarrollo de aplicaciones,NaN,Microsoft,módulo,".net, c#, visual studio code"


Pero por el momento se mantienen `subcategoria` y `tecnologias_mencionadas` para mejorar la categorización del contenido.

##  1.2 Limpieza inicial del texto

Se detectaron que los ultimos 8 registros tienen título en inglés. Se borraran para mantener el contenido en español.

In [6]:
df_mslearn.tail(8)

,titulo,texto,categoria,subcategoria,autor,tipo,tecnologias_mencionadas
4598,Architect production-grade multi-agent AI solu...,Learn how to design production-grade agentic A...,Inteligencia artificial,NaN,Microsoft,ruta de aprendizaje,"agent-framework, azure, azure-managed-redis, c..."
4599,"Monitor, evaluate, and operate multi-agent AI ...",Learn how to operate production multi-agent so...,Inteligencia artificial,NaN,Microsoft,ruta de aprendizaje,"agent-framework, azure, azure-managed-redis, c..."
4600,Minecraft Esports Teacher Academy,"Learn the role of esports in education, how to...",NaN,NaN,Microsoft,ruta de aprendizaje,minecraft
4601,Introduction to student security operations ce...,Learn what security operations centers (SOCs) ...,NaN,NaN,Microsoft,ruta de aprendizaje,"copilot, microsoft, microsoft sentinel, micros..."
4602,Introduction to AI literacy,Using the Empowering Learners for the Age of A...,NaN,NaN,Microsoft,ruta de aprendizaje,"ia, m365-education, microsoft, ms-copilot"
4603,Learning Accelerators for educators,Bring individualized learning to the classroom...,NaN,NaN,Microsoft,ruta de aprendizaje,"datos, desarrollo, learning, m365-education, m..."
4604,Troubleshoot Active Directory Domain Services ...,Diagnose Active Directory Domain Services repl...,Seguridad,Identidad y acceso,Microsoft,módulo,"data, windows server"
4605,From prompts to goals,Learn how to apply Copilot Cowork to everyday ...,Inteligencia artificial,NaN,Microsoft,módulo,"copilot, m365-apps, microsoft 365, ms-copilot"


In [7]:
df_mslearn = df_mslearn.iloc[:-8].reset_index(drop=True)

print(f"✅ Registros eliminados. Filas actuales: {len(df_mslearn)}")
display(df_mslearn.tail(3))

✅ Registros eliminados. Filas actuales: 4598


,titulo,texto,categoria,subcategoria,autor,tipo,tecnologias_mencionadas
4595,Diseño del Aprendizaje en el Siglo XXI,El Diseño del Aprendizaje en el Siglo XXI para...,NaN,NaN,Microsoft,ruta de aprendizaje,office 365
4596,Marco de Transformación Educativa,"Las formas en que las personas interactúan, so...",NaN,NaN,Microsoft,ruta de aprendizaje,"microsoft, office 365"
4597,Aprendizaje en contexto remoto con uso de Teams,El aprendizaje en contexto remoto se materiali...,NaN,NaN,Microsoft,ruta de aprendizaje,"microsoft teams, teams"


Se aplica la limpieza

In [8]:
def clean_html_urls(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    return re.sub(r'http\S+', '', text)

# Aplicar limpieza a las nuevas columnas del esquema
df_mslearn['titulo'] = df_mslearn['titulo'].apply(clean_html_urls)
df_mslearn['texto'] = df_mslearn['texto'].apply(clean_html_urls)
df_mslearn['categoria'] = df_mslearn['categoria'].apply(clean_html_urls)
df_mslearn['subcategoria'] = df_mslearn['subcategoria'].apply(clean_html_urls)

# Eliminar duplicados basados en el contenido del 'texto'
initial_count = len(df_mslearn)
df_mslearn = df_mslearn.drop_duplicates(subset='texto').reset_index(drop=True)
print(f"Duplicados eliminados: {initial_count - len(df_mslearn)} (Filas restantes: {len(df_mslearn)})")

# Filtro de calidad: mínimo 100 palabras en la columna 'texto'
initial_count = len(df_mslearn)
df_mslearn = df_mslearn[df_mslearn['texto'].str.split().str.len() >= 100].reset_index(drop=True)
print(f"Textos con menos de 100 palabras eliminados: {initial_count - len(df_mslearn)} (Filas restantes: {len(df_mslearn)})")

print("\u2705 Limpieza de HTML, URLs, duplicados y textos cortos completada.")

Duplicados eliminados: 14 (Filas restantes: 4584)
Textos con menos de 100 palabras eliminados: 3782 (Filas restantes: 802)
✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.


##  1.3 EDA

Cuantas filas hay por categoría

In [9]:
df_mslearn['categoria'].value_counts()

categoria
Aplicaciones empresariales    238
nan                           190
Seguridad                     103
Infraestructura                79
Desarrollo de aplicaciones     76
Inteligencia artificial        59
Administración de datos        57
Name: count, dtype: int64

La categoría no da información suficiente para la categorización que se busca.

---

¿Cuál es la Frecuencia por subcategoría?

In [10]:
df_mslearn['subcategoria'].value_counts()

subcategoria
nan                                                435
DevOps                                              34
Administración de la cadena de suministro           28
Computación en la nube                              26
Finanzas y contabilidad                             26
Marketing y ventas                                  22
Bases de datos                                      20
Identidad y acceso                                  15
Desarrollo de aplicaciones personalizadas           14
Comunicación                                        13
Architecture                                        12
Procesos de fabricación                             11
Proceso y flujo de trabajo                          10
Seguridad en la nube                                10
Administración del conocimiento                     10
Gobernanza y protección de la información            8
Análisis de datos                                    8
Gestión y monitoreo de TI                           

Tenemos subcategorías variadas y el caso donde no se registra subcategoría. 
Esto nos da más información.

Veamos el top de tecnologías mencionadas individualmente.

In [11]:
individual_techs = df_mslearn['tecnologias_mencionadas'].str.split(',').explode().str.strip()
print("--- Top 20 tecnologías individuales más mencionadas ---")
print(individual_techs.value_counts().head(20))

--- Top 20 tecnologías individuales más mencionadas ---
tecnologias_mencionadas
microsoft                       446
azure                           273
datos                           259
dynamics 365                    190
dynamics                        161
copilot                         139
microsoft 365                   129
ia                              109
desarrollo                       90
teams                            74
supply chain management          68
github                           63
microsoft teams                  63
windows                          61
office 365                       60
finance                          50
web                              48
ms-copilot                       43
business central                 42
microsoft defender for cloud     41
Name: count, dtype: int64


Al ser un dataset extraido de Microsoft Learn es normal que aparezcan los productos de esta empresa.

##  1.4.Mapeo de categorías (Lógica basada en Título, Texto, Tecnologías y Subcategoría)

In [12]:
# Diccionario 1: Palabras clave para búsqueda (Prioridad 1 y 2)
CATEGORY_KEYWORDS = {
    'Backend': ['java', 'spring', 'c#', 'php', 'node.js', 'django', 'perl', 'ruby', 'scala', 'clojure', 'rust', 'haskell', 'elixir', 'earlang', 'flask', 'typescript', 'node', 'nodejs', 'laravel', 'api', 'rest', 'backend', 'csharp', 'dotnet', 'blockchain', 'cplusplus', 'c', 'go', 'express', 'graphql', 'kafka', 'solr', 'rabbitMQ', 'nginx', 'openresty', 'nestjs', 'firebase', '.net', 'rails'],
    'Bases de Datos': ['sql', 'mysql', 'mongodb', 'postgresql', 'redis', 'oracledb', 'cassandra', 'couchdb', 'hive', 'realm', 'mariadb', 'cockroachdb', 'elasticsearch', 'sqlite', 'mssql', 'sql server', 'sqlserver', 'cosmos db', 'database', 'bases de datos', 'nosql', 'storage'],
    'Cloud': ['aws', 'azure', 'oraclecloud', 'oci', 'googlecloud', 'gcp', 'nube', 'cloud', 'máquina virtual', 'virtual machine'],
    'Data Science': ['python', 'pandas', 'machine learning', 'deep', 'excel', 'llm', 'r', 'ia', 'deep learning', 'tensorflow', 'pytorch', 'numpy', 'seaborn', 'matplotlib', 'opencv', 'scikitlearn', 'scikit learn', 'd3js', 'chartjs', 'canvasjs', 'kibana', 'grafana', 'inteligencia artificial', 'datos', 'analytics', 'data analysis', 'aprendizaje automático', 'modelado de datos'],
    'DevOps': ['docker', 'kubernetes', 'ci/cd', 'devops', 'git', 'github', 'jenkins', 'bash', 'travisci', 'circleci', 'containers', 'automation'],
    'Frontend': ['javascript', 'html', 'html5', 'css', 'css3', 'react', 'angular', 'angularjs', 'vue', 'vuejs', 'scratch', 'frontend', 'svelte', 'backbonejs', 'bootstrap', 'vuetify', 'pug', 'gulp', 'sass', 'redux', 'webpack', 'babel', 'tailwind', 'materialize', 'bulma', 'gtk', 'qt', 'quasar', 'wxwidgets', 'wx widgets', 'ember', 'web'],
    'Mobile': ['objectivec', 'objective-c', 'android', 'ios', 'flutter', 'kotlin', 'swift', 'dart', 'nativescript', 'xamarin', 'reactnative', 'react native', 'ionic', 'apachecordova', 'mobile', 'multiplataforma']
}

# Diccionario 2: Mapeo directo por columna 'subcategoria' (Prioridad 3)
SUBCAT_TO_CATEGORY = {
    'Architecture': 'DevOps',
    'DevOps': 'DevOps',
    'Computación en la nube': 'Cloud',
    'Seguridad en la nube': 'Cloud',
    'Bases de datos': 'Bases de Datos',
    'Análisis de datos': 'Data Science',
    'Modelado de datos': 'Data Science',
    'Visualización de datos': 'Data Science',
    'Aprendizaje automático': 'Data Science',
    'Integración de datos': 'Data Science',
    'Ingeniería de datos': 'Data Science',
    'Protección contra amenazas': 'DevOps',
    'containers': 'DevOps'
}

def assign_final_category(row):
    # 1. Búsqueda en Título y Texto (Prioridad 1)
    combined_text = (str(row['titulo']) + " " + str(row['texto'])).lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        for word in keywords:
            if re.search(r'\b' + re.escape(word) + r'\b', combined_text):
                return category

    # 2. Búsqueda en Tecnologías Mencionadas (Prioridad 2)
    techs_list = [t.strip().lower() for t in str(row['tecnologias_mencionadas']).split(',') if t.strip()]
    for category, keywords in CATEGORY_KEYWORDS.items():
        if any(kw in techs_list for kw in keywords):
            return category

    # 3. Mapeo por Subcategoría (Prioridad 3)
    subcat = str(row['subcategoria'])
    if subcat in SUBCAT_TO_CATEGORY:
        return SUBCAT_TO_CATEGORY[subcat]

    return None

# Aplicar el nuevo mapeo sin usar la columna 'categoria' original
df_mslearn['categoria'] = df_mslearn.apply(assign_final_category, axis=1)

# Diagnóstico: tasa de coincidencia
total_rows = len(df_mslearn)
matched_rows = df_mslearn['categoria'].notna().sum()
print(f" Categorías mapeadas con éxito: {matched_rows} de {total_rows} ({matched_rows/total_rows*100:.1f}%)" if total_rows > 0 else "No hay datos para procesar.")

if total_rows > 0 and (matched_rows / total_rows) < 0.3:
    print("⚠️ ALERTA: La tasa de coincidencia es menor al 30%. Revisa el diccionario de mapeo.")

df_mslearn = df_mslearn.dropna(subset=['categoria']).reset_index(drop=True)

print("\n✅ Categorías asignadas.")
print("--- Distribución resultante tras el mapeo ---")
print(df_mslearn['categoria'].value_counts())


 Categorías mapeadas con éxito: 604 de 802 (75.3%)

✅ Categorías asignadas.
--- Distribución resultante tras el mapeo ---
categoria
Cloud             233
Data Science      225
Bases de Datos     52
Backend            45
DevOps             24
Frontend           21
Mobile              4
Name: count, dtype: int64


Eliminar columnas auxiliares para finalizar el esquema

In [13]:
df_mslearn = df_mslearn.drop(columns=['subcategoria', 'tecnologias_mencionadas'])

print("✅ Columnas eliminadas.")
print(f"Columnas restantes: {df_mslearn.columns.tolist()}")
display(df_mslearn.head(3))

✅ Columnas eliminadas.
Columnas restantes: ['titulo', 'texto', 'categoria', 'autor', 'tipo']


,titulo,texto,categoria,autor,tipo
0,Protección de las interacciones de Copilot de ...,Las herramientas de inteligencia artificial co...,Data Science,Microsoft,módulo
1,Compartir conocimientos dentro de los equipos,Aprenda a compartir conocimientos de forma efi...,Cloud,Microsoft,módulo
2,"Gobernanza, barreras de protección y operaciones",Los sistemas de agente deben funcionar dentro ...,Data Science,Microsoft,módulo


##  1.5. Balanceo de datos (Máximo 50 registros por categoría)

In [14]:
df_mslearn = pd.concat([
    group.sample(min(len(group), 50), random_state=42)
    for _, group in df_mslearn.groupby('categoria')
]).reset_index(drop=True)

print("✅ Datos balanceados (Máximo 50 registros por categoría).")

✅ Datos balanceados (Máximo 50 registros por categoría).


##  1.6. Exportación final y auditoría de calidad

In [15]:
final_df_mslearn = df_mslearn.copy()

# Selecciona solo las columnas que forman parte del esquema final de la base de datos
final_df_mslearn = final_df_mslearn[['titulo', 'texto', 'categoria', 'autor', 'tipo']]

print("=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===")
print("   === DATASET - MICROSOFT LEARN ===")
category_counts = final_df_mslearn['categoria'].value_counts()
print(category_counts)

# Alerta si alguna categoría no alcanza el mínimo de 30 registros
print("\n--- Validación de Umbral Mínimo ---")
for cat, count in category_counts.items():
    if count < 30:
        print(f"⚠️ ADVERTENCIA: '{cat}' tiene {count} registros (Por debajo del mínimo de 30). Se requiere extracción manual.")

# Verifica que no haya columnas duplicadas antes de exportar
if final_df_mslearn.columns.duplicated().any():
    print("\n🚨 ERROR: hay columnas duplicadas en final_df_mslearn:")
    print(final_df_mslearn.columns[final_df_mslearn.columns.duplicated()].tolist())
else:
    print("\n✅ Sin columnas duplicadas — esquema limpio.")


final_df_mslearn.to_csv(f'{CARPETA_PROCESADOS}/dataset_FINAL_mslearn.csv', index=False)
print("\n Pipeline completado. Dataset exportado con éxito al formato oficial.")

# Muestra 20 filas aleatorias para revisión manual de calidad
print("\n=== MUESTRA DE AUDITORÍA MANUAL (Revisión de 20 filas aleatorias) ===")
audit_sample = final_df_mslearn.sample(min(20, len(final_df_mslearn)), random_state=42)[['titulo', 'texto','categoria', 'autor', 'tipo']]
display(audit_sample)


=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===
   === DATASET - MICROSOFT LEARN ===
categoria
Bases de Datos    50
Cloud             50
Data Science      50
Backend           45
DevOps            24
Frontend          21
Mobile             4
Name: count, dtype: int64

--- Validación de Umbral Mínimo ---
⚠️ ADVERTENCIA: 'DevOps' tiene 24 registros (Por debajo del mínimo de 30). Se requiere extracción manual.
⚠️ ADVERTENCIA: 'Frontend' tiene 21 registros (Por debajo del mínimo de 30). Se requiere extracción manual.
⚠️ ADVERTENCIA: 'Mobile' tiene 4 registros (Por debajo del mínimo de 30). Se requiere extracción manual.

✅ Sin columnas duplicadas — esquema limpio.

 Pipeline completado. Dataset exportado con éxito al formato oficial.

=== MUESTRA DE AUDITORÍA MANUAL (Revisión de 20 filas aleatorias) ===


,titulo,texto,categoria,autor,tipo
24,Acceso a los eventos de calendario de un usuar...,Microsoft Graph proporciona acceso a los datos...,Backend,Microsoft,módulo
6,Implementación de las herramientas de depuraci...,En este módulo se exploran las herramientas y ...,Backend,Microsoft,módulo
153,Protección de las interacciones y entornos de ...,La inteligencia artificial está cambiando cómo...,Data Science,Microsoft,ruta de aprendizaje
211,Desarrollo remoto con Visual Studio Code,El desarrollo remoto proporciona ventajas como...,DevOps,Microsoft,ruta de aprendizaje
198,Defiéndase contra las ciberamenazas con XDR de...,Para obtener esta credencial de aptitudes apli...,DevOps,Microsoft,ruta de aprendizaje
176,Trabajar con el comercio entre empresas vincul...,Aprenda a trabajar con el comercio entre empre...,Data Science,Microsoft,módulo
192,Proyecto guiado: adoptar la IA responsable,Vaya más allá de la teoría y aprenda a impleme...,Data Science,Microsoft,módulo
124,Administración de datos de configuración de ap...,Obtenga información sobre los patrones de conf...,Cloud,Microsoft,módulo
9,"Herramientas, MCP y entornos de ejecución de a...",En este módulo se presenta cómo interactúan lo...,Backend,Microsoft,módulo
101,Ejercicio guiado: Administración de servidores...,"En este ejercicio guiado, practicará la incorp...",Cloud,Microsoft,módulo
